# AGN-gwfast Example Notebook

This is a little notebook illustrating how to interact with the new `NewGWSignal` and `AGNLensedGWSignal` classes.

In [1]:
from pathlib import Path

from jax import config
import jax.numpy as np
config.update("jax_enable_x64", True)

import numpy as onp

import gwfast.network as network
import gwfast.waveforms as waveforms
from gwfast.gwfastGlobals import detectors as det_dict, detPath
from gwfast.detector import Detector
from gwfast.AGN_lensed_signal import AGNLensedGWSignal
from gwfast.new_signal import NewGWSignal
import gwfast.fisherTools as fTools

/users/hin-wai.leong/src/AGN-gwfast/gwfast/waveforms.py:30: UserWarning: Wswiglal-redir-stdio:

SWIGLAL standard output/error redirection is enabled in IPython.
This may lead to performance penalties. To disable locally, use:

with lal.no_swig_redirect_standard_output_error():
    ...

To disable globally, use:

lal.swig_redirect_standard_output_error(False)

Note however that this will likely lead to error messages from
LAL functions being either misdirected or lost when called from
Jupyter notebooks.

To suppress this warning, use:

import warnings
warnings.filterwarnings("ignore", "Wswiglal-redir-stdio")
import lal

  import lal


TEOBResumS is not installed, only the GWFAST waveform models are available, namely: TaylorF2, IMRPhenomD, IMRPhenomD_NRTidalv2, IMRPhenomHM and IMRPhenomNSBH


In [2]:
events = {
    'Mc':np.array([30, 30]), 'eta':np.array([0.20, 0.24]), 'dL':np.array([2, 2]), 
    'theta':np.array([2.34, 2.34]), 'phi':np.array([5.43, 5.43]), 
    'iota':np.array([0.99*np.pi/2, 0.99*np.pi/2]), 'psi':np.array([1, 1]), 
    'tGPS':np.array([0, 0]), 'Phicoal':np.array([2.8, 2.8]), 
    'chi1z':np.array([1e-3, 1e-3]), 'chi2z':np.array([1e-3, 1e-3]), 
    'R_orbit':np.array([200, 50]), 'M_lz':np.array([10e4, 10e5]), 'src_pos':np.array([0.7, 0.1])
}
# This casting is to ensure all int becomes float
events = {key: val.astype(np.float64) for key, val in events.items()}

# Initialise the detector objects themselves
H1 = Detector('H1', **det_dict['H1'],
              noise_curve_path=Path(detPath)/'observing_scenarios_paper/AplusDesign.txt')
L1 = Detector('L1', **det_dict['L1'],
              noise_curve_path=Path(detPath)/'observing_scenarios_paper/AplusDesign.txt')
V1 = Detector('V1', **det_dict['Virgo'],
              noise_curve_path=Path(detPath)/'observing_scenarios_paper/avirgo_O5low_NEW.txt')

# Define some waveforms
PhenomD = waveforms.IMRPhenomD()
PhenomHM = waveforms.IMRPhenomHM()

## Signal initialisation

This step may take some time

In [3]:
# Define the detectors and networks with AGN signal
H1_AGN = AGNLensedGWSignal(wf_model=PhenomD, detector=H1, fmin=10)
L1_AGN = AGNLensedGWSignal(wf_model=PhenomD, detector=L1, fmin=10)
V1_AGN = AGNLensedGWSignal(wf_model=PhenomD, detector=V1, fmin=10)
HL_AGN_signals = network.DetNet({'H1': H1_AGN, 'L1': L1_AGN})
HLV_AGN_signals = network.DetNet({'H1': H1_AGN, 'L1': L1_AGN, 'V1': V1_AGN})

# Define the detectors and networks with vanilla BBH signal
H1_BBH = NewGWSignal(wf_model=PhenomD, detector=H1, fmin=10)
L1_BBH = NewGWSignal(wf_model=PhenomD, detector=L1, fmin=10)
V1_BBH = NewGWSignal(wf_model=PhenomD, detector=V1, fmin=10)
HL_BBH_signals = network.DetNet({'H1': H1_BBH, 'L1': L1_BBH})
HLV_BBH_signals = network.DetNet({'H1': H1_BBH, 'L1': L1_BBH, 'V1': V1_BBH})

Initializing jax...
Jax local device count: 8
Jax device count: 8
Initializing jax...
Jax local device count: 8
Jax device count: 8
Initializing jax...
Jax local device count: 8
Jax device count: 8
Initializing jax...
Jax local device count: 8
Jax device count: 8
Initializing jax...
Jax local device count: 8
Jax device count: 8
Initializing jax...
Jax local device count: 8
Jax device count: 8


## Compute the GW strain (only L1 for illustration)

In [4]:
# Define the frequency array to be evaluated on
N_freqs = 500
f_array = np.geomspace(20, 400, num=N_freqs)
# Broadcast the frequency array to the appropriate shape for multiple parameter values
f_array = np.broadcast_to(f_array, (events['Mc'].shape[0], N_freqs)).T

L1_BBH_strains = L1_BBH.GWstrain(f_array, events)
L1_AGN_strains = L1_AGN.GWstrain(f_array, events)
print(L1_BBH_strains.T.shape, L1_AGN_strains.T.shape)

(2, 500) (2, 500)


Notice that even though `events` contains the keys for lensing parameters, the vanilla `BBH` signal still happily accepts it.

This brings us to the first point, the new `GWstrain` function does not rely on any location parameters.
Moreover, as long as the necessary parameters for the chosen waveform model are supplied, it will happily compute the waveform.

In fact, notice there is no flags about which parameters are being inputed (*e.g.* `Mc, eta` or `m1, m2`; `ra, dec` or `theta, phi`), there is a new `get_model_parameters` function within the `GWstrain` function, which will get the needed parameters from the given ones, so long as they are sufficient. 

See the example below:

In [5]:
# Showing the flexibility of `get_model_parameters`

# For PhenomD, the required parameters are:
# Mc, eta, chi1z, chi2z, iota, Phicoal, theta, phi, psi, dL, tcoal (total: 11 params)

# In this example, we supply (m1, m2) and (ra, dec) to `GWStrain` instead of the canonical ones

from gwfast.gwfastUtils import m1m2_from_Mceta

Mc = events.pop('Mc')
eta = events.pop('eta')
events['m1'], events['m2'] = m1m2_from_Mceta(Mc, eta)

theta = events.pop('theta')
phi = events.pop('phi')
events['ra'] = phi
events['dec'] = np.pi / 2 - theta

print('Check that the new `events` has only the new keys: ', events.keys())

L1_BBH_strains_2 = L1_BBH.GWstrain(f_array, events)
L1_AGN_strains_2 = L1_AGN.GWstrain(f_array, events)

BBH_diff = np.abs(L1_BBH_strains - L1_BBH_strains_2)
AGN_diff = np.abs(L1_AGN_strains - L1_AGN_strains_2)

print('For BBH: ')
print(f'Max. diff.: {np.max(BBH_diff):.4e}')
print(f'Max. frac. diff.: {np.max(BBH_diff / np.abs(L1_BBH_strains)):.4e}')
print(f'Std. dev.:  {np.std(BBH_diff):.4e}')
print('For AGN: ')
print(f'Max. diff.: {np.max(AGN_diff):.4e}')
print(f'Max. frac. diff.: {np.max(AGN_diff / np.abs(L1_AGN_strains)):.4e}')
print(f'Std. dev.:  {np.std(AGN_diff):.4e}')

Check that the new `events` has only the new keys:  dict_keys(['dL', 'iota', 'psi', 'tGPS', 'Phicoal', 'chi1z', 'chi2z', 'R_orbit', 'M_lz', 'src_pos', 'm1', 'm2', 'ra', 'dec'])
For BBH: 
Max. diff.: 1.1761e-39
Max. frac. diff.: 3.1513e-15
Std. dev.:  1.7545e-40
For AGN: 
Max. diff.: 7.1325e-39
Max. frac. diff.: 1.2482e-14
Std. dev.:  6.0936e-40


So these two are practically the same, only differ in machine precision.

## The fun part, Fisher matrices

Similar to how `GWstrain` is upgraded to accept any given parameters, so are the Fisher matrices. 

Not only that, previously, the Fisher matrices will only return the matrices for a preset list of parameters, now it will return the matrix for whatever given parameters, **in the same order as the parameter dictionary**.

In [6]:
fisher_mats_BBH = HLV_BBH_signals.FisherMatr(events)
keys = list(events.keys())

print("\nThe Fisher matrices:")
# Swapping the axes so that it can be iterated over the different sets of parameters.
fisher_mats_iter = np.moveaxis(fisher_mats_BBH, 2, 0)
for matrix in fisher_mats_iter:
    row = f'{"":8}   ' + '  '.join([f'{col_key:^11}' for col_key in keys])
    print(row)
    for rdx, row_key in enumerate(keys):
        row = f'{row_key:>8}  '
        for cdx, col_key in enumerate(keys):
            row += f'{matrix[rdx][cdx]:+11.3e}  '
        print(row)
    print('--------------------')

Computing Fisher for H1...
Computing Fisher for L1...
Computing Fisher for V1...
Done.

The Fisher matrices:
               dL          iota          psi         tGPS        Phicoal       chi1z        chi2z       R_orbit       M_lz        src_pos        m1           m2           ra           dec    
      dL   +1.475e+01   +1.651e+00   +3.292e+00   -1.003e-03   +2.518e-17   -7.525e+00   -2.232e+00   +0.000e+00   +0.000e+00   +0.000e+00   -3.995e-02   -8.466e-01   +1.376e+01   -2.215e+01  
    iota   +1.651e+00   +9.235e+01   -2.069e-01   +2.975e+03   -6.585e+00   +2.914e+02   +4.723e+01   +0.000e+00   +0.000e+00   +0.000e+00   -1.560e+01   -3.632e+01   -6.264e+01   -1.066e+02  
     psi   +3.292e+00   -2.069e-01   +9.259e+01   -2.937e+03   +5.156e+00   -2.578e+02   -3.678e+01   +0.000e+00   +0.000e+00   +0.000e+00   +1.300e+01   +2.955e+01   +9.903e+01   -1.579e+02  
    tGPS   -1.003e-03   +2.975e+03   -2.937e+03   +2.660e+07   -3.453e+04   +1.871e+06   +1.873e+05   +0.000e+00   +0.00

As promised, it will compute the matrix for any given parameters. Since the lensing-related parameters have no effect on the waveform, and they are returned with entire column/row of zeroes, as expected.

In [7]:
# Now, use the reduce_Fisher_matrix function to get rid of the zeros.
# Note that, in this case, the zeros are expected and safe to remove.
# But sometimes unexpected zeroes imply something is wrong in the waveform/parameters.
alt_keys = keys.copy()
reduced_fisher_mats_BBH, _ = fTools.reduce_Fisher_matrix(fisher_mats_BBH, alt_keys)

print("\nThe Reduced Fisher matrices:")
# Swapping the axes so that it can be iterated over the different sets of parameters.
fisher_mats_iter = np.moveaxis(reduced_fisher_mats_BBH, 2, 0)
for matrix in fisher_mats_iter:
    row = f'{"":8}   ' + '  '.join([f'{col_key:^11}' for col_key in alt_keys])
    print(row)
    for rdx, row_key in enumerate(alt_keys):
        row = f'{row_key:>8}  '
        for cdx, col_key in enumerate(alt_keys):
            row += f'{matrix[rdx][cdx]:+11.3e}  '
        print(row)
    print('--------------------')


The Reduced Fisher matrices:
               dL          iota          psi         tGPS        Phicoal       chi1z        chi2z         m1           m2           ra           dec    
      dL   +1.475e+01   +1.651e+00   +3.292e+00   -1.003e-03   +2.518e-17   -7.525e+00   -2.232e+00   -3.995e-02   -8.466e-01   +1.376e+01   -2.215e+01  
    iota   +1.651e+00   +9.235e+01   -2.069e-01   +2.975e+03   -6.585e+00   +2.914e+02   +4.723e+01   -1.560e+01   -3.632e+01   -6.264e+01   -1.066e+02  
     psi   +3.292e+00   -2.069e-01   +9.259e+01   -2.937e+03   +5.156e+00   -2.578e+02   -3.678e+01   +1.300e+01   +2.955e+01   +9.903e+01   -1.579e+02  
    tGPS   -1.003e-03   +2.975e+03   -2.937e+03   +2.660e+07   -3.453e+04   +1.871e+06   +1.873e+05   -9.014e+04   -2.025e+05   -1.167e+05   -5.822e+04  
 Phicoal   +2.518e-17   -6.585e+00   +5.156e+00   -3.453e+04   +5.901e+01   -2.976e+03   -4.150e+02   +1.502e+02   +3.422e+02   +1.534e+02   +7.618e+01  
   chi1z   -7.525e+00   +2.914e+02   -2.578e+02

### Finally, the Covariance matrix

In [13]:
print('=================================================================')
cov_mats, ie = fTools.CovMatr(reduced_fisher_mats_BBH)
covar_mats_iter = onp.moveaxis(cov_mats, 2, 0)

print("\nThe Covariance matrices:")
for matrix in covar_mats_iter:
    row = f'{"":8}   ' + '  '.join([f'{col_key:^11}' for col_key in alt_keys])
    print(row)
    for rdx, row_key in enumerate(alt_keys):
        row = f'{row_key:>8}  '
        for cdx, col_key in enumerate(alt_keys):
            row += f'{matrix[rdx][cdx]:+11.3e}  '
        print(row)


The Covariance matrices:
               dL          iota          psi         tGPS        Phicoal       chi1z        chi2z         m1           m2           ra           dec    
      dL   +1.460e-01   +2.295e-03   +5.073e-02   +9.766e-03   +2.291e-01   -2.417e-01   +6.641e-01   -3.666e+00   +1.265e+00   -2.608e-02   +1.707e-02  
    iota   +2.295e-03   +1.333e-02   +5.957e-03   -2.883e-04   +1.708e-03   +7.202e-03   -1.871e-02   +7.407e-02   -2.184e-02   -1.255e-03   +2.768e-03  
     psi   +5.073e-02   +5.957e-03   +8.536e-02   +4.399e-04   +4.322e-02   -1.340e-02   +3.574e-02   -1.746e-01   +4.981e-02   -3.565e-02   +2.413e-02  
    tGPS   +9.766e-03   -2.883e-04   +4.399e-04   +1.349e-02   +1.257e+00   -3.326e-01   +8.860e-01   -3.716e+00   +9.579e-01   -2.588e-04   +1.914e-04  
 Phicoal   +2.291e-01   +1.708e-03   +4.322e-02   +1.257e+00   +1.403e+02   -3.124e+01   +8.210e+01   -3.420e+02   +8.064e+01   -2.414e-02   +1.929e-02  
   chi1z   -2.417e-01   +7.202e-03   -1.340e-02   -

In [17]:
for idx, matrix in enumerate(covar_mats_iter):
    print(f'\nFor parameter set {idx+1:d}')
    for key, val in zip(alt_keys, onp.sqrt(onp.diag(matrix))):
        print(key, '--', val)


For parameter set 1
dL -- 0.38214532277045435148
iota -- 0.11545570679457918084
psi -- 0.2921627706614417413
tGPS -- 0.11615645620689913362
Phicoal -- 11.845202172303719692
chi1z -- 2.867288526530024902
chi2z -- 7.629221319020975774
m1 -- 33.454225372953681424
m2 -- 9.01642369393040612
ra -- 0.13611723688279695668
dec -- 0.0906336621412510968

For parameter set 2
dL -- 0.3431174231624196416
iota -- 0.10575973417374696936
psi -- 0.26447260565452001497
tGPS -- 0.08087741003496006942
Phicoal -- 9.3364118332715690714
chi1z -- 6.7828527808584770514
chi2z -- 10.34807031503748437
m1 -- 42.534938152958847536
m2 -- 24.560049644100675962
ra -- 0.12269605284657910972
dec -- 0.08155999720477187975
